**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/masset-2025"], check=True)
    os.chdir("/content/encore/entries/masset-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Masset et al. (2025): VR task, common reward expectation model (DANDI:000251)

**Paper.** Masset P, Tano P, Kim HR, Malik AN, Pouget A, Uchida N (2025). Multi-timescale
reinforcement learning in the brain. *Nature* 642:682-690. doi:10.1038/s41586-025-08929-9.
The claims file (`claims/masset-2025.yaml`) was written from the accepted author manuscript in
PubMed Central (main text, legends, Methods and Extended Data legends), with the paper's Mendeley
code (doi:10.17632/tc43t3s7c5.1) and simulation code (github.com/pablotano8/multi_timescale_RL at
37f11c6).

**Data.** DANDI:000251, version `draft`, modified 2023-06-20T00:56:22Z. The dandiset has no
published version, so the draft is the only version there is; it is pinned by its modification
time, which the notebook checks. It holds the data of Kim et al. (2020), which the paper reuses:
the VR task single units, one unit per file, and no cued delay task recordings. The cued delay
units are in the paper's own deposit, Mendeley Data doi:10.17632/tc43t3s7c5.1 (version 1,
2025-04-09), file `DA_SU_ODOR_NEW_W_OL.h5`; its `FinalAverageArray.mat` (the paper's per-neuron
table) and `Valid_S.mat` (the paper's neuron flags) link the two tasks' units. All files are
streamed into memory.

**Seeds.** Random draws use each claim's seed, `zlib.crc32(b"masset-2025/<claim id>")`, and four
more (plus 1000 to 4000); a second generator of a claim adds 1. Shared computations use the seed
of the claim that defines them: the VR half splits that of `fig3b-vr-ramp-population`, the VR fit's
initial values that of `fig3-common-value-fit`, the cued delay fits that of `cued-delay-neurons`.

The entry has four notebooks: `vr_population.ipynb` (VR claims on the population),
`vr_single_mouse.ipynb` (single-mouse VR fits), `vr_reward_expectation.ipynb` (the common reward
expectation model), `cued_delay_and_cross_task.ipynb` (link, cued delay and cross-task claims). The claims each one tests are listed below.

**This notebook.** Tested: ed9g-models-agree. It refits the common value model at the claims'
seeds (as `vr_population.ipynb` does) to compare with. The reward expectation fits are
initialised from the seed of `ed9g-models-agree`; the half splits are those of
`fig3b-vr-ramp-population`.


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
import time
import zlib
import collections
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import multiscale as ms

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
T0 = time.time()
OFFSETS = [0, 1000, 2000, 3000, 4000]


def seed(claim, offset=0):
    return zlib.crc32(f"masset-2025/{claim}".encode()) + offset


# --- load
info = ms.dandi_version_info()
print("dandiset", ms.DANDISET, "version", ms.DANDI_VERSION, "modified", info["modified"])
assert info["modified"].startswith(ms.DANDI_MODIFIED[:19])
assets = ms.dandi_assets()
files = ms.dandi_su_files(assets, kinds=("SU-VR-OL",))
print(f"loaded in {time.time() - T0:.0f} s")

## Data check

The open-loop units and the M6 selection, as in `vr_population.ipynb`, where the files are
checked in full, and the common value fits to compare with.


In [ ]:
vr, vr_dropped = ms.dandi_vr_units(files)
vr_rates = [ms.vr_trial_rates(u) for u in vr]
vr_mean = np.array([r.mean() for r in vr_rates])
vr_ml = np.array([u["ml"] for u in vr])
vr_sel = (vr_mean > ms.RATE_MIN) & (vr_ml < ms.ML_MAX)
rates_sel = [r for r, s in zip(vr_rates, vr_sel) if s]
n_vr = len(rates_sel)
print(f"open-loop units {len(vr)} (left out: {vr_dropped}); selected (M6): {n_vr}")

N_SUB = 10  # bootstraps fitted under the four other seeds (20 fits each)
t = time.time()
VR = {}
for off in OFFSETS:
    ps = ms.vr_bootstrap_psths(rates_sel, np.random.default_rng(seed("fig3b-vr-ramp-population", off)))
    if off != 0:
        ps = ps[:N_SUB]
    x0 = ms.cv_initial(n_vr, np.random.default_rng(seed("fig3-common-value-fit", off)))
    VR[off] = dict(ps=ps, fits=ms.cv_fit_bootstraps(ps, x0))
f = VR[0]["fits"]
print(f"common value fits (200 at the claims' seeds, 20 under each other seed): {time.time() - t:.0f} s; "
      f"claims' seeds gamma mean {f['gamma'].mean(0).mean():.3f}, s.d. {f['gamma'].mean(0).std(ddof=1):.3f}")


### ed9g-models-agree

> Discount factors from the common value model and the common reward expectation model are
> strongly correlated across neurons (rho = 0.93).

The common reward expectation model (Methods eq. 28 to 31; code Cost_VR_FoldedNormal_All.m,
FoldedNormal.m, ValueConstraint.m): each neuron's value is its discount factor applied to a folded
normal belief over the time to reward (horizon 10 s), whose width is free at each bin and shared
across neurons; the mean value over neurons is at most 2 at every bin. It is fitted by SLSQP with
analytic gradients (at most 2000 iterations, as the code's fmincon), from the code's initial
values, to the same half PSTHs as the common value model. Under the claims' seeds all 200 halves
are fitted; under the four other seeds, the first 10 bootstraps (20 fits), compared with the
common value fits of the same 20 halves. Spearman correlation of the per-neuron mean discount
factors, two-tailed.


In [ ]:
from scipy import optimize
R = VR[0]["ps"][0, 0]
x0f = ms.fr_initial(n_vr, np.random.default_rng(seed("ed9g-models-agree")))
f_, g_ = ms.fr_sse(x0f, R)
num = optimize.approx_fprime(x0f, lambda x: ms.fr_sse(x, R)[0], 1e-6)
Jn = np.array([optimize.approx_fprime(x0f, lambda x: ms.fr_constraint(x, n_vr)[t], 1e-7) for t in range(ms.VR_NBINS)])
print(f"analytic against numerical gradient: largest difference {np.abs(num - g_).max() / np.abs(g_).max():.1e} "
      f"of the largest component; constraint Jacobian {np.abs(Jn - ms.fr_constraint_jac(x0f, n_vr)).max() / np.abs(Jn).max():.1e}")

FR = {}
t = time.time()
for off in OFFSETS:
    ps = VR[off]["ps"]
    nb = ps.shape[0]
    x0f = ms.fr_initial(n_vr, np.random.default_rng(seed("ed9g-models-agree", off)))
    FR[off] = ms.fr_fit_halves([ps[k % nb, k // nb] for k in range(2 * nb)], x0f)
print(f"reward expectation fits: {time.time() - t:.0f} s")
rows = []
for off in OFFSETS:
    fr, cv = FR[off], VR[off]["fits"]
    sub = np.r_[0:N_SUB, 100:100 + N_SUB] if off == 0 else np.arange(2 * N_SUB)
    row = dict(seed=off, fits=len(fr["ok"]), converged=int(fr["ok"].sum()),
               iterations_median=np.median(fr["nit"]), constraint_min=fr["cmin"].min(),
               gamma_mean=fr["gamma"].mean(0).mean(), gamma_sd=fr["gamma"].mean(0).std(ddof=1),
               at_lower_bound=int((fr["gamma"].mean(0) < 0.06).sum()))
    r, p = stats.spearmanr(fr["gamma"].mean(0), cv["gamma"].mean(0))
    row.update(rho_all_fits=r, p_all_fits=p)
    r, p = stats.spearmanr(fr["gamma"][sub].mean(0), cv["gamma"][sub].mean(0))
    row.update(rho_20_fits=r, p_20_fits=p)
    rows.append(row)
E9 = pd.DataFrame(rows)
print(E9.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("expected: Spearman rho 0.93 +- 0.093 (200 fits each); rho_20_fits is over the first 10 bootstraps")
print("(constraint_min: smallest 2 minus mean value over the fits, negative if the constraint is broken;"
      " at_lower_bound: neurons whose mean discount factor is below 0.06, near the bound 0.05)")
g_fr, g_cv = FR[0]["gamma"].mean(0), VR[0]["fits"]["gamma"].mean(0)
print(f"claims' seeds: reward expectation gamma {g_fr.min():.3f} to {g_fr.max():.3f}; common value gamma "
      f"{g_cv.min():.3f} to {g_cv.max():.3f}; Pearson r {stats.pearsonr(g_fr, g_cv)[0]:.3f}")
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(g_cv, g_fr, s=10)
ax.set_xlabel("discount factor, common value model")
ax.set_ylabel("discount factor, reward expectation model")
plt.show()


**Result: ed9g-models-agree.** Under the claims' seeds, 187 of the 200 reward expectation fits converged (the constraint holds in every fit to within 1.1e-6), and the per-neuron mean discount factors of the two models have Spearman rho = 0.743 (p = 3.3e-17, 91 neurons), below 0.93 +- 0.093. Under the four other seeds, on 20 fits each, rho is 0.769 to 0.809, also below. The discount factors of the two models are strongly correlated, less than reported: partially reproduced. The reward expectation model puts 17 neurons at a mean discount factor below 0.06, at the lower bound (mean 0.218, against 0.446 for the common value model). The result rests on the iteration cap of the common value fit and on SLSQP in place of fmincon.


## Summary

| Claim | Outcome (claims' seeds) |
|---|---|
| ed9g-models-agree | partially reproduced, resting on the iteration cap of the common value fit |

The outcomes under the other four seeds are in the table above and in the attempt record.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method. The reward expectation model is fitted by SLSQP with
analytic gradients, at most 2000 iterations, where the code uses fmincon (interior point). The
folded normal density is written as a sum of two Gaussians, equal to the code's form, which
overflows at small widths (the code then sets the value to 1e-4). Under the four other seeds only
the first 10 bootstraps are fitted. The common value fits are those of `vr_population.ipynb`
(L-BFGS-B, 10,000 iterations, an assumption).
